## Ejercicio Guiado: Misión a Kepler-186f

## Mapa de Rutas y Costos

In [ ]:
# Grafo con costos de combustible.
rutas_espaciales = {
  'Tierra': {'Marte': 4, 'Estacion Alfa': 3},
  'Marte': {'Jupiter': 5},
  'Estacion Alfa': {'Jupiter': 2, 'Cinturon Asteroides': 6},
  'Jupiter': {'Saturno': 3},
  'Cinturon Asteroides': {'Kepler-186f': 7},
  'Saturno': {'Kepler-186f': 4},
  'Kepler-186f': {}
}

## Función Heurística `h(n)`

In [ ]:
# Distancia estimada a Kepler-186f
heuristica = {
  'Tierra': 10, 'Marte': 8,
  'Estacion Alfa': 9, 'Jupiter': 6,
  'Cinturon Asteroides': 5, 'Saturno': 3,
  'Kepler-186f': 0
}

## Paso 2: El Software de Navegación

In [ ]:
import heapq

# --- Implementación de Búsqueda Voraz Primero el Mejor ---
def greedy_best_first(graph, start, goal, heuristics):
    # La cola de prioridad ordena los nodos solo por su valor heurístico.
    priority_queue = [(heuristics[start], [start])] # (h(n), camino)
    visited = set()

    while priority_queue:
        h, path = heapq.heappop(priority_queue)
        node = path[-1]

        if node in visited: continue
        visited.add(node)
        
        if node == goal: return path

        for neighbor in graph[node]:
            if neighbor not in visited:
                heapq.heappush(priority_queue, (heuristics[neighbor], path + [neighbor]))
    return None

# --- Implementación de Búsqueda A* (A-Estrella) ---
def a_star_search(graph, start, goal, heuristics):
    # f(n) = g(n) + h(n)
    # g(n) es el costo real del camino desde el inicio.
    # h(n) es el costo estimado hasta el final.
    priority_queue = [(0 + heuristics[start], 0, [start])] # (f(n), g(n), camino)
    visited = set()

    while priority_queue:
        f, g, path = heapq.heappop(priority_queue)
        node = path[-1]

        if node in visited: continue
        visited.add(node)

        if node == goal: return path, g

        for neighbor, cost in graph[node].items():
            if neighbor not in visited:
                new_g = g + cost
                new_f = new_g + heuristics[neighbor]
                heapq.heappush(priority_queue, (new_f, new_g, path + [neighbor]))
    return None, -1

## Paso 3: Ejecutar los Algoritmos e Imprimir Resultados

In [ ]:
INICIO = 'Tierra'
META   = 'Kepler-186f'

# --- Ejecutar Greedy Best-First ---
ruta_greedy = greedy_best_first(rutas_espaciales, INICIO, META, heuristica)

# Calcular costo real de la ruta Greedy (la función no lo devuelve)
costo_greedy = sum(
    rutas_espaciales[ruta_greedy[i]][ruta_greedy[i+1]]
    for i in range(len(ruta_greedy) - 1)
)

print('=' * 55)
print('  🚀 MISIÓN A KEPLER-186f — RESULTADOS DE BÚSQUEDA')
print('=' * 55)

print('\n📡 Búsqueda Voraz (Greedy Best-First):')
print(f'   Ruta encontrada : {" → ".join(ruta_greedy)}')
print(f'   Costo real      : {costo_greedy} unidades de combustible')
print(f'   Pasos           : {len(ruta_greedy) - 1}')

# --- Ejecutar A* ---
ruta_astar, costo_astar = a_star_search(rutas_espaciales, INICIO, META, heuristica)

print('\n⭐ Búsqueda A* (A-Estrella):')
print(f'   Ruta encontrada : {" → ".join(ruta_astar)}')
print(f'   Costo real      : {costo_astar} unidades de combustible')
print(f'   Pasos           : {len(ruta_astar) - 1}')

print('\n' + '=' * 55)
if costo_astar < costo_greedy:
    print(f'  ✅ A* encontró una ruta más eficiente ({costo_astar} vs {costo_greedy})')
elif costo_astar == costo_greedy:
    print(f'  ✅ Ambos algoritmos encontraron la misma ruta óptima (costo {costo_astar})')
else:
    print(f'  ⚠️  Greedy fue más eficiente esta vez ({costo_greedy} vs {costo_astar})')
print('=' * 55)

## Paso 4: Visualización del Mapa Espacial y las Rutas

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Construir el grafo dirigido ---
G = nx.DiGraph()
for origen, destinos in rutas_espaciales.items():
    for destino, costo in destinos.items():
        G.add_edge(origen, destino, weight=costo)

# --- Posiciones fijas para una disposición clara de izquierda a derecha ---
pos = {
    'Tierra':              (0,  0),
    'Marte':               (2,  1),
    'Estacion Alfa':       (2, -1),
    'Jupiter':             (4,  1),
    'Cinturon Asteroides': (4, -1),
    'Saturno':             (6,  1),
    'Kepler-186f':         (8,  0),
}

# --- Helper: extraer aristas de una ruta ---
def ruta_a_aristas(ruta):
    return list(zip(ruta[:-1], ruta[1:]))

aristas_greedy = ruta_a_aristas(ruta_greedy)
aristas_astar  = ruta_a_aristas(ruta_astar)

# --- Colores de nodos ---
def color_nodo(nodo, ruta):
    if nodo == INICIO:        return '#00e5ff'   # cian — origen
    if nodo == META:          return '#69ff47'   # verde — destino
    if nodo in ruta:          return '#ffb300'   # ámbar — en ruta
    return '#546e7a'                              # gris — no visitado

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0d0d1a')

titulos    = ['📡 Greedy Best-First', '⭐ A* (A-Estrella)']
rutas      = [ruta_greedy, ruta_astar]
aristas    = [aristas_greedy, aristas_astar]
costos     = [costo_greedy, costo_astar]
colores_ruta = ['#ff4081', '#7c4dff']

edge_labels = nx.get_edge_attributes(G, 'weight')

for ax, titulo, ruta, arista_ruta, costo, color_r in zip(
        axes, titulos, rutas, aristas, costos, colores_ruta):

    ax.set_facecolor('#0d1117')

    colores_nodos = [color_nodo(n, ruta) for n in G.nodes()]

    # Dibujar todas las aristas en gris tenue
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edge_color='#37474f', width=1.5,
        arrows=True, arrowsize=18,
        connectionstyle='arc3,rad=0.08'
    )

    # Resaltar aristas de la ruta encontrada
    nx.draw_networkx_edges(
        G, pos, edgelist=arista_ruta, ax=ax,
        edge_color=color_r, width=4,
        arrows=True, arrowsize=25,
        connectionstyle='arc3,rad=0.08'
    )

    # Dibujar nodos
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=colores_nodos, node_size=1800,
        edgecolors='white', linewidths=1.5
    )

    # Etiquetas de nodos
    nx.draw_networkx_labels(
        G, pos, ax=ax,
        font_color='white', font_size=8, font_weight='bold'
    )

    # Etiquetas de costos en aristas
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels=edge_labels, ax=ax,
        font_color='#ffeb3b', font_size=9,
        bbox=dict(boxstyle='round,pad=0.2', fc='#0d1117', ec='none', alpha=0.7)
    )

    ax.set_title(
        f'{titulo}\nRuta: {" → ".join(ruta)}\nCosto: {costo} combustible',
        color='white', fontsize=11, fontweight='bold', pad=12
    )
    ax.axis('off')

    # Leyenda
    leyenda = [
        mpatches.Patch(color='#00e5ff', label='Origen (Tierra)'),
        mpatches.Patch(color='#69ff47', label='Meta (Kepler-186f)'),
        mpatches.Patch(color='#ffb300', label='Nodo en ruta'),
        mpatches.Patch(color='#546e7a', label='Nodo no visitado'),
        mpatches.Patch(color=color_r,   label='Ruta encontrada'),
    ]
    ax.legend(
        handles=leyenda, loc='lower left',
        facecolor='#1a1a2e', edgecolor='#37474f',
        fontsize=8, labelcolor='white'
    )

fig.suptitle(
    '🚀 Misión a Kepler-186f — Comparación de Algoritmos de Búsqueda Informada',
    color='white', fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('mision_kepler.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('\n✅ Gráfica guardada como mision_kepler.png')